# 01 — Contracts playground

Boundary-fuzz every Pydantic model in `src/hsb/contracts/`. Every model declares `model_config = {"extra": "forbid"}` — so any LLM-emitted field that isn't in the schema must fail parse.

**No LLM, no MCP. Pure Pydantic.** Identical results on Claude or Codex — the contracts validate after the runtime hands back a final JSON blob, so the runtime has no say.

What you can do here: poke fields, see exactly which validator fires, learn which constraints are hard (Pydantic) vs soft (system prompt only).


In [ ]:
from _helpers import ensure_src_on_path

ensure_src_on_path()
from pydantic import ValidationError

## extra='forbid' — every contract rejects rogue fields

If you ever see a contract drop this constraint, the LLM gains a silent escape hatch. Cheap, blanket check.


In [ ]:
import importlib
import inspect

from pydantic import BaseModel

modules = [
    "hsb.contracts.qa",
    "hsb.contracts.knowledge",
    "hsb.contracts.risk",
    "hsb.contracts.uat",
    "hsb.contracts.global_orchestrator",
    "hsb.contracts.main_orchestrator",
    "hsb.contracts.orchestrator",
    "hsb.contracts.backlog",
    "hsb.contracts.builder",
    "hsb.contracts.git",
    "hsb.contracts.linear",
]
missing = []
for mname in modules:
    m = importlib.import_module(mname)
    for name, obj in inspect.getmembers(m, inspect.isclass):
        if obj.__module__ != mname:
            continue
        if not issubclass(obj, BaseModel):
            continue
        cfg = getattr(obj, "model_config", {}) or {}
        if cfg.get("extra") != "forbid":
            missing.append(f"{mname}.{name}")
assert not missing, f"contracts missing extra='forbid': {missing}"
print("every contract model declares extra=forbid")

## QA — cycle cap (Pitfall 2)

Three branches: cap-reached + still-required (BLOCK), cap-reached + approved + missing annotation (BLOCK), canonical cap-reached path (PASS).


In [ ]:
from hsb.contracts.qa import QAEvidence, QAFinding, QAOutput

ev = QAEvidence(file="a.py", component="c", location="10", related_requirement="AC-1")
f = QAFinding(
    title="t",
    severity="low",
    category="code_quality",
    status="non_blocking",
    problem="p",
    evidence=ev,
    expected_behavior="e",
    actual_behavior="a",
    suggested_fix="s",
)

# 5 findings = OK, 6 = blocked by max_length
QAOutput(
    work_item_id="LIN-1",
    qa_status="changes_required",
    qa_cycle_count=1,
    summary="s",
    findings=[f] * 5,
)
raised = False
try:
    QAOutput(
        work_item_id="LIN-1",
        qa_status="changes_required",
        qa_cycle_count=1,
        summary="s",
        findings=[f] * 6,
    )
except ValidationError:
    raised = True
assert raised, "findings cap (max 5) not enforced"
print("QAAG-03: max_length=5 on findings enforced")

In [ ]:
from hsb.contracts.qa import PullRequestInput, QAInput

pr = PullRequestInput(url="https://github.com/x/y/pull/1", diff="--- a\n+++ b\n")
QAInput(work_item_id="LIN-1", linear_issue={}, pull_request=pr, qa_cycle_count=2)
raised = False
try:
    QAInput(work_item_id="LIN-1", linear_issue={}, pull_request=pr, qa_cycle_count=3)
except ValidationError:
    raised = True
assert raised, "QAInput accepted qa_cycle_count > 2 (input is 0-indexed, max 2)"
print("QAInput: input qa_cycle_count is 0-indexed [0..2]")

## UAT — evidence min_length (B2 dimension)

`UATScenario.evidence` has `min_length=10` so the LLM cannot pass off paraphrases of the criterion as evidence.


In [ ]:
from hsb.contracts.uat import UATResult, UATScenario

raised = False
try:
    UATScenario(
        criterion_id="AC-1", criterion_text="t", status="pass", evidence="short"
    )
except ValidationError:
    raised = True
assert raised, "UAT evidence accepted len < 10"
UATScenario(
    criterion_id="AC-1",
    criterion_text="t",
    status="pass",
    evidence="ten chars or more here yes",
)

# uat_cycle is 1-indexed (ge=1)
raised = False
try:
    UATResult(
        user_story_id="US-1", overall_status="approved", scenarios=[], uat_cycle=0
    )
except ValidationError:
    raised = True
assert raised, "UATResult accepted uat_cycle=0"
print("UAT: evidence min_length=10 + uat_cycle ge=1 both enforced")

## Linear — entity ID + URL regex; failed-result invariant

`LinearEntity.id` matches `^LIN-\d+$`; URL must be `https://linear.app/...`. `LinearOutput.failed_must_have_error` model_validator forbids a failed result without an error message.


In [ ]:
from hsb.contracts.linear import LinearEntity, LinearOutput

raised = False
try:
    LinearEntity(id="123", type="task", url="https://linear.app/x")
except ValidationError:
    raised = True
assert raised, "LinearEntity accepted id without LIN- prefix"

raised = False
try:
    LinearOutput(operation="create", result="failed", linear_entities=[], error=None)
except ValidationError:
    raised = True
assert raised, "failed result without error message accepted"

ok = LinearOutput(operation="create", result="failed", linear_entities=[], error="boom")
assert ok.error == "boom"
print("Linear: id regex, URL regex, failed-must-have-error all enforced")

## Knowledge Store — applicability validator (G9, INTL-03)

Same fuzz as notebook 00, with a twist: confirm trimming + case folding both apply (`'  ALL  '` should still be rejected).


In [ ]:
from hsb.contracts.knowledge import KnowledgeStorageInput

base = dict(
    title="t",
    type="qa",
    context="c",
    evidence={
        "linear_issue": "LIN-1",
        "pr": "#1",
        "files": ["x.py"],
        "qa_finding": "F1",
    },
    insight="i",
    recommendation="r",
    date="2026-05-09",
)
for bad in ["  ALL  ", "\tn/a\n", "TBD", "   "]:
    raised = False
    try:
        KnowledgeStorageInput(**base, applicability=bad)
    except Exception:
        raised = True
    assert raised, f"whitespace/case fold bypass: {bad!r}"
print("G9: trimming + case fold both apply")

## Risk — score range and Literal pin


In [ ]:
from hsb.contracts.risk import AutoImprovementTrigger, QualityScore

raised = False
try:
    QualityScore(work_item_id="LIN-1", score=101.0)
except ValidationError:
    raised = True
assert raised, "QualityScore accepted score > 100"

raised = False
try:
    QualityScore(work_item_id="LIN-1", score=-1.0)
except ValidationError:
    raised = True
assert raised, "QualityScore accepted negative score"

raised = False
try:
    AutoImprovementTrigger(
        title="t",
        description="d",
        pattern_evidence=["LIN-1", "LIN-2"],
        suggested_type="x",
        linear_state="created",
    )
except ValidationError:
    raised = True
assert raised, "AutoImprovementTrigger accepted linear_state=created"
print("Risk: score 0..100 + linear_state Literal[suggested] both pinned")

## Backlog — empty epics list rejected; plan_source required


In [ ]:
from hsb.contracts.backlog import (
    BacklogInput,
    BacklogOutput,
    BacklogTraceability,
    ProjectContext,
)

raised = False
try:
    BacklogOutput(epics=[], traceability=BacklogTraceability(plan_source="plan.md"))
except ValidationError:
    raised = True
assert raised, "BacklogOutput accepted empty epics list"

raised = False
try:
    BacklogInput(
        project_context=ProjectContext(name="x", repository="y")
    )  # missing plan_source
except ValidationError:
    raised = True
assert raised, "BacklogInput accepted missing plan_source (BKPK-01)"
print("Backlog: min_length=1 on epics + plan_source required (BKPK-01)")

## Builder — capability boundary at the schema level (BLDR-04)

If the agent emits `git_branch` / `pr_url` / `linear_status`, `extra='forbid'` rejects.


In [ ]:
from hsb.contracts.builder import BuilderOutput

for forbidden in ["git_branch", "pr_url", "linear_status"]:
    payload = dict(
        work_item_id="LIN-1",
        implementation_status="completed",
        summary="s",
        files_changed=[],
        validation={
            "build": "not_run",
            "tests": "not_run",
            "lint": "not_run",
            "typecheck": "not_run",
        },
        implementation_notes={},
    )
    payload[forbidden] = "leaked"
    raised = False
    try:
        BuilderOutput.model_validate(payload)
    except ValidationError:
        raised = True
    assert raised, f"BuilderOutput accepted {forbidden} field"
print("BLDR-04: BuilderOutput rejects git_branch / pr_url / linear_status fields")

## Git — capability boundary at the schema level (GITA-05)


In [ ]:
from hsb.contracts.git import GitOutput, PullRequest

pr = PullRequest(
    url="https://github.com/x/y/pull/1",
    title="[LIN-1] add x",
    base="epic/LIN-99",
    head="feature/LIN-1-add-x",
)
for forbidden in ["merged_to_main", "linear_status", "file_changes"]:
    payload = dict(
        work_item_id="LIN-1",
        branch="feature/LIN-1-add-x",
        commits=[],
        pull_request=pr.model_dump(),
    )
    payload[forbidden] = "leaked"
    raised = False
    try:
        GitOutput.model_validate(payload)
    except ValidationError:
        raised = True
    assert raised, f"GitOutput accepted {forbidden} field"
print("GITA-05: GitOutput rejects merge / linear / code-change fields")

## Free-form play

Use the cell below to instantiate any contract you're investigating. The cell intentionally has no assertions so you can observe the validator's error messages directly.


In [ ]:
# Example: try editing one field at a time and re-running.
from hsb.contracts.qa import QAOutput

try:
    out = QAOutput(
        work_item_id="LIN-1",
        qa_status="approved",
        qa_cycle_count=1,
        summary="ok",
        findings=[],
    )
    print(out.model_dump())
except ValidationError as e:
    print(e.json(indent=2))